# Step 6: Hooks — 런타임 확장 메커니즘

## 학습 목표

- Claude Code가 **~20개 라이프사이클 포인트**에서 동작을 커스터마이징하는 방법을 이해합니다
- Hook이 **셸 명령으로 실행**되어 언어 독립적인 이유를 배웁니다
- **PreToolUse/PostToolUse** 훅이 권한 시스템과 연결되는 방식을 이해합니다

> **Hooks란?**
> 에이전트의 주요 동작 시점(도구 호출 전/후, 세션 시작/종료 등)에 사용자 정의 로직을 삽입하는 메커니즘입니다.
> 플러그인 시스템의 일종으로, 에이전트 코어를 수정하지 않고도 동작을 확장할 수 있습니다.

---

## Claude Code 소스 분석

### 6-1. 훅이 실행되는 시점 — 라이프사이클 포인트

Claude Code는 에이전트 동작의 주요 시점마다 훅을 실행합니다:

```
사용자 입력
    │
    ▼
┌──────────────┐
│ SESSION_START │ ← 세션 시작 시 환경 검증
└──────┬───────┘
       ▼
  while (true) {
       │
       ▼
  ┌────────────┐
  │ LLM 응답    │
  └────┬───────┘
       ▼
  ┌──────────────────┐
  │ PRE_TOOL_USE     │ ← 도구 실행 전: 차단/수정 가능
  │  (권한 체크 전!)   │
  └──────┬───────────┘
       ▼
  ┌──────────────┐
  │ 도구 실행      │
  └──────┬───────┘
       ▼
  ┌──────────────────┐
  │ POST_TOOL_USE    │ ← 도구 실행 후: 로깅/알림
  └──────┬───────────┘
       ▼
  }
       │
       ▼
┌──────────────┐
│ SESSION_END  │ ← 세션 종료 시 정리 작업
└──────────────┘
```

**핵심 패턴: PreToolUse 훅은 권한 체크(permission) 전에 실행됩니다.**
이것은 훅이 권한 시스템보다 먼저 실행되어 도구 호출을 조기 차단할 수 있다는 의미입니다.

### 6-2. Hook 타입 정의 (~20 라이프사이클 포인트) (hooks.ts)

```typescript
// src/types/hooks.ts — Hook 타입 정의

type HookType =
  | "PreToolUse"       // 도구 실행 전 (차단/수정 가능)
  | "PostToolUse"      // 도구 실행 후 (로깅/알림)
  | "Notification"     // 알림 발생 시
  | "Stop"             // 에이전트 정지 시
  // ... 약 20개 라이프사이클 포인트

type HookConfig = {
  matcher: string;     // 도구 이름 패턴 (glob: "Bash", "File*", "*")
  command: string;     // 실행할 셸 명령어
};

// settings.json에서 선언:
// {
//   "hooks": {
//     "PreToolUse": [
//       { "matcher": "Bash", "command": "check-safety.sh" }
//     ]
//   }
// }
```

### 6-3. 훅 실행 엔진 (utils/hooks.ts)

```typescript
// src/utils/hooks.ts — executeHooks()

async function executeHooks(
  hookType: HookType,
  toolName: string,
  input: any,
): Promise<HookResult> {
  const hooks = getMatchingHooks(hookType, toolName);

  for (const hook of hooks) {
    // 셸 명령어 실행 (언어 독립적!)
    const result = await runShellCommand(hook.command, {
      env: {
        HOOK_TOOL_NAME: toolName,
        HOOK_TOOL_INPUT: JSON.stringify(input),
      },
    });

    // 종료 코드로 결과 판단:
    //   0 → ALLOW (계속 진행)
    //   2 → DENY  (도구 실행 차단)
    if (result.exitCode === 2) {
      return { action: "deny", message: result.stderr };
    }
  }
  return { action: "allow" };
}
```

**셸 명령어를 사용하는 이유:**
- 어떤 프로그래밍 언어로도 훅을 작성할 수 있음 (Python, Bash, Go, ...)
- 기존 CLI 도구를 훅으로 직접 사용 가능 (eslint, prettier, ...)
- 환경변수로 컨텍스트를 전달하므로 인터페이스가 단순함

### 6-4. PreToolUse/PostToolUse 호출 위치 (toolExecution.ts)

```typescript
// src/services/tools/toolExecution.ts — 도구 실행 파이프라인

async function executeTool(toolCall, context) {
  // 1. PreToolUse 훅 실행 ← 권한 체크 전!
  const preHookResult = await executeHooks("PreToolUse", toolCall.name, toolCall.input);
  if (preHookResult.action === "deny") {
    return { error: preHookResult.message };  // 조기 차단
  }
  if (preHookResult.action === "modify") {
    toolCall.input = preHookResult.modifiedInput;  // 입력 수정
  }

  // 2. 권한 체크 (permission system — Step 7에서 다룸)
  const permitted = await checkPermission(toolCall);
  if (!permitted) return { error: "Permission denied" };

  // 3. 도구 실행
  const result = await tool.call(toolCall.input, context);

  // 4. PostToolUse 훅 실행
  await executeHooks("PostToolUse", toolCall.name, {
    input: toolCall.input,
    result: result,
  });

  return result;
}
```

---

## Python으로 구현하기

`mini_claude/hooks.py`에 구현된 훅 시스템을 확인합니다.

### 핵심 클래스 구조

1. **HookType** — 훅이 실행되는 시점 (Enum)
2. **HookAction / HookResult** — 훅 실행 결과 (ALLOW, DENY, MODIFY)
3. **Hook** — 훅 정의 (matcher + command 또는 handler)
4. **HookRegistry** — 훅 등록 및 실행 엔진

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.hooks import (
    HookType, HookAction, HookResult, Hook, HookRegistry,
)

# --- HookType: 라이프사이클 시점 ---
print("=== HookType (라이프사이클 포인트) ===")
for ht in HookType:
    print(f"  {ht.value:20s} - {ht.name}")
print()

# --- Hook: 매칭 패턴 확인 ---
print("=== Hook 매칭 패턴 (glob) ===")
hook = Hook(type=HookType.PRE_TOOL_USE, matcher="Bash", command="echo test")
test_names = ["Bash", "Read", "Write", "BashExec"]
for name in test_names:
    print(f"  matcher='Bash', tool='{name}' -> {hook.matches(name)}")

print()
hook_glob = Hook(type=HookType.PRE_TOOL_USE, matcher="File*", command="echo test")
test_names2 = ["FileRead", "FileWrite", "Bash", "FileEdit"]
for name in test_names2:
    print(f"  matcher='File*', tool='{name}' -> {hook_glob.matches(name)}")

print()
hook_all = Hook(type=HookType.PRE_TOOL_USE, matcher=None, command="echo test")
print(f"  matcher=None (모든 도구 매칭): tool='Bash' -> {hook_all.matches('Bash')}")
print(f"  matcher=None (모든 도구 매칭): tool='Read' -> {hook_all.matches('Read')}")

### 실습 1: PreToolUse 훅 — Bash 명령어 로깅

모든 Bash 도구 호출을 로깅하는 PreToolUse 훅을 등록합니다.
Python callable 훅을 사용하여 도구 호출 전에 어떤 명령이 실행되는지 기록합니다.

In [ ]:
import asyncio
from datetime import datetime

# --- 훅 레지스트리 생성 ---
registry = HookRegistry()

# --- 1. PreToolUse 훅: Bash 명령어 로깅 ---
bash_log = []  # 로그 저장소

async def log_bash_commands(context: dict) -> HookResult:
    """모든 Bash 도구 호출의 명령어를 로깅합니다."""
    tool_input = context.get("tool_input", {})
    command = tool_input.get("command", "(no command)")
    timestamp = datetime.now().strftime("%H:%M:%S")
    log_entry = f"[{timestamp}] Bash: {command}"
    bash_log.append(log_entry)
    print(f"  [PreToolUse Hook] {log_entry}")
    return HookResult(action=HookAction.ALLOW)

registry.register(Hook(
    type=HookType.PRE_TOOL_USE,
    matcher="Bash",            # Bash 도구에만 매칭
    handler=log_bash_commands,
))

# --- 2. PreToolUse 훅: 위험한 명령어 차단 ---
DANGEROUS_PATTERNS = ["rm -rf /", "mkfs", "dd if=", ":(){:|:&};:"]

async def block_dangerous_commands(context: dict) -> HookResult:
    """위험한 Bash 명령어를 차단합니다."""
    tool_input = context.get("tool_input", {})
    command = tool_input.get("command", "")
    for pattern in DANGEROUS_PATTERNS:
        if pattern in command:
            print(f"  [PreToolUse Hook] BLOCKED: '{command}' (pattern: '{pattern}')")
            return HookResult(
                action=HookAction.DENY,
                message=f"Dangerous command blocked: contains '{pattern}'",
            )
    return HookResult(action=HookAction.ALLOW)

registry.register(Hook(
    type=HookType.PRE_TOOL_USE,
    matcher="Bash",
    handler=block_dangerous_commands,
))

# --- 테스트: 훅 실행 ---
print("=== PreToolUse 훅 테스트 ===\n")

# 안전한 명령어 → ALLOW
print("1. 안전한 명령어:")
result = await registry.execute_hooks(
    HookType.PRE_TOOL_USE,
    tool_name="Bash",
    context={"tool_input": {"command": "echo hello"}},
)
print(f"  결과: {result.action.value}\n")

# 위험한 명령어 → DENY
print("2. 위험한 명령어:")
result = await registry.execute_hooks(
    HookType.PRE_TOOL_USE,
    tool_name="Bash",
    context={"tool_input": {"command": "rm -rf / --no-preserve-root"}},
)
print(f"  결과: {result.action.value}, 메시지: {result.message}\n")

# Read 도구 → 훅이 매칭되지 않음 (matcher="Bash")
print("3. Read 도구 (Bash 훅이 매칭되지 않음):")
result = await registry.execute_hooks(
    HookType.PRE_TOOL_USE,
    tool_name="Read",
    context={"tool_input": {"file_path": "/etc/passwd"}},
)
print(f"  결과: {result.action.value} (매칭되는 훅 없음 → 기본 ALLOW)\n")

# 로그 확인
print("=== Bash 명령어 로그 ===")
for entry in bash_log:
    print(f"  {entry}")

### 실습 2: PostToolUse 훅 — Write 후 자동 git add

파일을 쓴 후 자동으로 `git add`를 실행하는 PostToolUse 훅을 만듭니다.
이런 패턴은 실제로 CI/CD 파이프라인에서 자주 사용됩니다.

In [ ]:
import subprocess, tempfile, os

# --- PostToolUse 훅: Write 후 자동 git add ---
git_add_log = []

async def auto_git_add(context: dict) -> HookResult:
    """Write 도구 실행 후 자동으로 git add를 실행합니다."""
    tool_input = context.get("tool_input", {})
    file_path = tool_input.get("file_path", "")
    if file_path:
        # 실제로는 subprocess.run(["git", "add", file_path])를 호출하지만,
        # 여기서는 시뮬레이션합니다
        log_entry = f"git add {file_path}"
        git_add_log.append(log_entry)
        print(f"  [PostToolUse Hook] {log_entry}")
    return HookResult(action=HookAction.ALLOW)

# 새 레지스트리 생성
registry2 = HookRegistry()
registry2.register(Hook(
    type=HookType.POST_TOOL_USE,
    matcher="Write",           # Write 도구에만 매칭
    handler=auto_git_add,
))

# --- 셸 명령어 훅도 테스트 ---
# Claude Code 방식: command로 셸 명령어를 지정
registry2.register(Hook(
    type=HookType.POST_TOOL_USE,
    matcher="*",               # 모든 도구에 매칭
    command="echo '[PostToolUse Shell Hook] Tool: $HOOK_TOOL_NAME'",
))

print("=== PostToolUse 훅 테스트 ===\n")

# Write 도구 시뮬레이션
print("1. Write 도구 실행 후:")
result = await registry2.execute_hooks(
    HookType.POST_TOOL_USE,
    tool_name="Write",
    context={"tool_input": {"file_path": "/tmp/test.py", "content": "print('hello')"}},
)
print(f"  결과: {result.action.value}\n")

# Bash 도구 → Write 훅은 매칭 안 됨, 셸 훅만 실행
print("2. Bash 도구 실행 후 (Write 훅 매칭 안 됨, 셸 훅만 실행):")
result = await registry2.execute_hooks(
    HookType.POST_TOOL_USE,
    tool_name="Bash",
    context={"tool_input": {"command": "ls"}},
)
print(f"  결과: {result.action.value}\n")

# 여러 Write 실행 후 로그 확인
print("3. 여러 파일 Write 시뮬레이션:")
for path in ["/tmp/a.py", "/tmp/b.py", "/tmp/c.py"]:
    await registry2.execute_hooks(
        HookType.POST_TOOL_USE,
        tool_name="Write",
        context={"tool_input": {"file_path": path, "content": "..."}},
    )

print(f"\n=== git add 로그 (총 {len(git_add_log)}건) ===")
for entry in git_add_log:
    print(f"  {entry}")

### 실습 3: MODIFY 액션 — 도구 입력 수정

PreToolUse 훅은 ALLOW/DENY뿐 아니라 **MODIFY**도 반환할 수 있습니다.
이를 통해 도구 입력을 수정하여 전달할 수 있습니다 (예: 경로 정규화, 인자 보정).

In [ ]:
# --- MODIFY 액션: Bash 명령어에 타임아웃 자동 추가 ---

async def add_timeout_to_bash(context: dict) -> HookResult:
    """Bash 명령어에 'timeout 30' 접두사를 자동 추가합니다."""
    tool_input = context.get("tool_input", {})
    command = tool_input.get("command", "")

    # 이미 timeout이 있으면 무시
    if command.startswith("timeout "):
        return HookResult(action=HookAction.ALLOW)

    # timeout 30 접두사 추가
    modified = {**tool_input, "command": f"timeout 30 {command}"}
    print(f"  [MODIFY Hook] '{command}' -> '{modified['command']}'")

    return HookResult(
        action=HookAction.MODIFY,
        modified_input=modified,
    )

# 레지스트리에 등록
registry3 = HookRegistry()
registry3.register(Hook(
    type=HookType.PRE_TOOL_USE,
    matcher="Bash",
    handler=add_timeout_to_bash,
))

print("=== MODIFY 액션 테스트 ===\n")

# 일반 명령어 → timeout 자동 추가
print("1. 일반 명령어:")
context = {"tool_input": {"command": "sleep 100"}}
result = await registry3.execute_hooks(
    HookType.PRE_TOOL_USE, tool_name="Bash", context=context,
)
print(f"  action: {result.action.value}")
print(f"  수정된 입력: {result.modified_input}")
print()

# 이미 timeout이 있는 명령어 → 수정 안 함
print("2. 이미 timeout이 있는 명령어:")
context2 = {"tool_input": {"command": "timeout 10 sleep 5"}}
result2 = await registry3.execute_hooks(
    HookType.PRE_TOOL_USE, tool_name="Bash", context=context2,
)
print(f"  action: {result2.action.value}")
print(f"  수정된 입력: {result2.modified_input}")
print()

# --- 훅 실행 순서: 여러 훅이 순차 실행됨 ---
print("=== 훅 실행 순서 (등록 순서대로) ===\n")

order_log = []

async def hook_a(ctx):
    order_log.append("A")
    print(f"  [Hook A] 실행됨 (순서: {len(order_log)})")
    return HookResult(action=HookAction.ALLOW)

async def hook_b(ctx):
    order_log.append("B")
    print(f"  [Hook B] 실행됨 (순서: {len(order_log)})")
    return HookResult(action=HookAction.ALLOW)

async def hook_c(ctx):
    order_log.append("C")
    print(f"  [Hook C] 실행됨 (순서: {len(order_log)})")
    return HookResult(action=HookAction.ALLOW)

registry4 = HookRegistry()
for handler in [hook_a, hook_b, hook_c]:
    registry4.register(Hook(type=HookType.PRE_TOOL_USE, handler=handler))

await registry4.execute_hooks(HookType.PRE_TOOL_USE, tool_name="Bash")
print(f"\n실행 순서: {' -> '.join(order_log)}")
print("훅은 등록 순서대로 순차 실행됩니다. DENY가 반환되면 나머지는 건너뜁니다.")

---

## 핵심 설계 원칙 정리

### 1. 라이프사이클 훅으로 코어 코드 수정 없이 확장
에이전트 루프의 주요 시점에 훅 포인트를 두면, 사용자가 에이전트 코어 코드를 건드리지 않고도 동작을 커스터마이징할 수 있습니다. 이것은 프레임워크의 핵심 패턴(Open-Closed Principle)입니다.

### 2. 셸 명령어 = 언어 독립적 확장
Claude Code의 훅은 셸 명령어로 실행됩니다. Python, Bash, Go, Rust 등 어떤 언어로든 훅을 작성할 수 있고, 기존 CLI 도구(eslint, prettier, git 등)를 직접 훅으로 사용할 수 있습니다. 종료 코드 0=ALLOW, 2=DENY라는 단순한 규칙만 지키면 됩니다.

### 3. 훅이 권한 체크보다 먼저 실행
PreToolUse 훅은 권한 시스템(Permission)보다 먼저 실행됩니다. 이 순서 덕분에 훅이 도구 실행을 조기 차단하거나 입력을 수정할 수 있고, 이후 권한 체크는 수정된 입력에 대해 수행됩니다.

### 4. Glob 패턴으로 유연한 매칭
`matcher`에 glob 패턴을 사용하여 여러 도구를 한 번에 매칭할 수 있습니다. `"Bash"`는 정확히 Bash만, `"File*"`는 FileRead/FileWrite/FileEdit 모두, `"*"`는 모든 도구에 매칭됩니다.

---

## 다음 Step 미리보기

**Step 7: Permission System**에서는 훅과 권한 시스템이 어떻게 통합되는지 배웁니다:
- **AllowAll / DenyAll / AllowList** 등 권한 정책 패턴
- 훅의 DENY 결과가 **권한 거부**와 동일하게 처리되는 방식
- 사용자에게 **승인 프롬프트**를 표시하는 인터랙티브 권한 체크